Purpose: Plot diel gene expression profiles for CA-cycle, photorespiration, and N metabolism gene families.<br>
Author: Anna Pardo<br>
Date initiated: June 30, 2026

In [1]:
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

In [17]:
# load pathway annotation
path = pd.read_csv("./photresp_N_citrate_genes_Yucca.csv",sep=",",header="infer")
path.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [2]:
# load Yg TPM
mdtpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",sep="\t",header="infer")
mdtpm.head()

/tmp/ipykernel_33787/2639278733.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  mdtpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",sep="\t",header="infer")


,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y1,18,1.0,W,1.0,gloriosa,34.002815,4.546167,0.0,17.236119,...,0.000000,6.167441,1.435253,0.279077,11.420139,0.662252,1.886709,6.675105,0.0,0.000000
1,Y10,2AB,1.5,W,3.0,gloriosa,40.070758,3.628454,0.0,14.918115,...,0.713535,2.197522,15.695904,1.193256,10.172780,0.000000,2.214483,12.125756,0.0,0.000000
2,Y100,2AB,6.5,W,23.0,gloriosa,47.402599,5.201760,0.0,17.406497,...,0.314746,2.261806,21.742515,1.798380,10.256681,1.748663,2.075757,25.215640,0.0,1.838780
3,Y101,2AB,1.0,D,1.0,gloriosa,57.062380,6.374324,0.0,10.567561,...,0.000000,1.072366,27.573939,1.164591,9.105769,1.257431,2.431447,4.508369,0.0,0.813682
4,Y103,2AB,1.0,D,1.0,gloriosa,34.679279,6.087451,0.0,11.115252,...,0.000000,0.914550,28.621412,0.869052,7.076224,0.515567,2.419222,4.625880,0.0,0.867418


In [3]:
mdtpm["genotype"] = mdtpm["genotype"].astype(str)
mdtpm["genotype"].unique()

array(['18', '2AB', '1AB', '19', '15', 'Eudy', 'G', '56', '36', '13',
       '45', '52', '43', '37', '48', '55', '70', '61', '51', '46', '53',
       '16', '6', '12', '20', '50'], dtype=object)

In [4]:
# load parental TPM
yatpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_reanalysis_30-Jun-2025/TPM/Yalo_TPM_withmd_July2025.txt",sep="\t",header="infer")
yftpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_reanalysis_30-Jun-2025/TPM/YfilH1_TPM_withmd_July2025.txt",sep="\t",header="infer")

In [5]:
phys = json.load(open("./physiology/physiological_categories_from_TA.json"))

In [6]:
mdtpm["CAM physiology"] = mdtpm["genotype"].map(phys)

In [7]:
yatpm["CAM physiology"] = "CAM"
yftpm["CAM physiology"] = "C3"

In [8]:
zttpm = mdtpm[mdtpm["ZT"].isin([1.0,5.0,9.0,13.0,17.0,21.0])]

In [21]:
# functions for plotting: modified from previous notebook, CAM_TS_genes.ipynb
def plottpm_genefamily(tpm, genefam, annot=path, add_phys=True):
    
    fam = annot[annot["gene_family"] == genefam].copy()

    # genes present in this TPM matrix
    genes = [g for g in fam["GeneID"].unique() if g in tpm.columns]

    if len(genes) == 0:
        return pd.DataFrame()

    cols = ["sample_name", "genotype", "treat", "ZT"] + genes
    pwide = tpm[cols].copy()

    plong = pd.melt(
        pwide,
        id_vars=["sample_name", "genotype", "treat", "ZT"],
        value_vars=genes,
        var_name="GeneID",
        value_name="TPM"
    )

    # map metadata
    subdict = fam.set_index("GeneID")["subgenome"].to_dict()
    namedict = fam.set_index("GeneID")["gene_name_unique"].to_dict()

    plong["subgenome"] = plong["GeneID"].map(subdict)
    plong["genename"] = plong["GeneID"].map(namedict)

    if add_phys:
        plong["CAM physiology"] = plong["genotype"].map(phys)

    return plong

In [22]:
def build_family_stylemaps(genefam,annot=path, yfpal="copper",yapal="winter"):

    fam = annot[annot["gene_family"] == genefam].copy()

    genes = fam["gene_name_unique"].unique().tolist()
    yfgenes = [i for i in genes if i.startswith("Yf")]
    yagenes = [i for i in genes if i.startswith("Ya")]

    yf_palette = sns.color_palette(yfpal, n_colors=max(len(yfgenes), 3))
    ya_palette = sns.color_palette(yapal, n_colors=max(len(yagenes), 3))
    color_map = {**dict(zip(yfgenes, yf_palette)), **dict(zip(yagenes,ya_palette))}

    # treatment line styles
    treat_map = {
        "W": "",
        "D": (4, 2)
    }

    return color_map, treat_map

In [14]:
def plot_profile(df, ax, title, color_map, treat_map,
                 log=False, fontsize=18):

    if df.empty:
        ax.set_title(title, fontsize=fontsize)
        return

    plotdf = df.copy()

    ycol = "TPM"

    if log:
        plotdf["logTPM"] = np.log1p(plotdf["TPM"])
        ycol = "logTPM"

    sns.lineplot(
        data=plotdf,
        x="ZT",
        y=ycol,
        hue="genename",        # each gene gets color
        style="treat",        # treatment gets linestyle
        palette=color_map,
        dashes=treat_map,
        estimator="mean",
        errorbar=None,
        lw=2,
        ax=ax
    )

    # dark period shading
    t = np.arange(0, 25)
    ax.fill_between(
        t, 0, 1,
        where=t >= 12,
        color="gray",
        alpha=0.25,
        transform=ax.get_xaxis_transform()
    )

    ax.set_xlim(0, 24)
    ax.set_xticks(sorted(plotdf["ZT"].unique()))
    ax.set_xlabel("")
    ax.set_ylabel("TPM")
    ax.set_title(title, fontsize=fontsize)
    ax.tick_params(labelsize=14)

In [16]:
def plot_genefam_withparents(genefam, d="./", save=True, log=False):

    # load data
    yaplot = plottpm_genefamily(yatpm, genefam, add_phys=False)
    yfplot = plottpm_genefamily(yftpm, genefam, add_phys=False)
    gftpm  = plottpm_genefamily(zttpm, genefam, add_phys=True)

    # physiology groups
    c3  = gftpm[gftpm["CAM physiology"] == "C3+CAM"]
    fac = gftpm[gftpm["CAM physiology"] == "facultative CAM"]
    cam = gftpm[gftpm["CAM physiology"] == "CAM"]

    c3gt  = c3["genotype"].nunique()
    facgt = fac["genotype"].nunique()
    camgt = cam["genotype"].nunique()

    # style maps
    color_map, treat_map = build_family_stylemaps(genefam)

    # figure
    fig, ax = plt.subplots(
        nrows=1,
        ncols=5,
        figsize=(30, 6),
        sharey=True
    )

    plot_profile(yfplot, ax[0],
                 "Y. filamentosa (C3 parent)",
                 color_map, treat_map, log, 18)

    plot_profile(c3, ax[1],
                 f"C3+CAM genotypes (n={c3gt})",
                 color_map, treat_map, log, 18)

    plot_profile(fac, ax[2],
                 f"Facultative CAM (n={facgt})",
                 color_map, treat_map, log, 18)

    plot_profile(cam, ax[3],
                 f"CAM genotypes (n={camgt})",
                 color_map, treat_map, log, 18)

    plot_profile(yaplot, ax[4],
                 "Y. aloifolia (CAM parent)",
                 color_map, treat_map, log, 18)

    # unify y-axis
    ymax = max(a.get_ylim()[1] for a in ax)
    for a in ax:
        a.set_ylim(0, ymax)

    # legend from middle panel
    handles, labels = ax[1].get_legend_handles_labels()

    for a in ax:
        leg = a.get_legend()
        if leg:
            leg.remove()

    fig.legend(
        handles,
        labels,
        loc="upper right",
        bbox_to_anchor=(1, 1.1),
        ncol=min(len(labels), 6),
        fontsize=14,
        frameon=False
    )

    plt.suptitle(genefam, fontsize=24)
    fig.tight_layout()

    if save:
        if not os.path.exists(d):
            os.makedirs(d)
        
        if "/" in genefam:
            genefam = genefam.split("/")[0]+"-"+genefam.split("/")[1]
            
        base = os.path.join(d, f"{genefam}_profiles_by_treatment")
        plt.savefig(base + ".png", dpi=250, bbox_inches="tight")
        plt.savefig(base + ".pdf", dpi=250, bbox_inches="tight")
        plt.savefig(base + ".svg", dpi=250, bbox_inches="tight")

In [18]:
direc = "./pathway_profiles"

In [23]:
for i in path["gene_family"].unique():
    plot_genefam_withparents(i,direc)
    plt.cla()
    plt.clf()

/tmp/ipykernel_33787/1044680728.py:21: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(


<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

...

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>

<Figure size 2160x432 with 0 Axes>